[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/12.%20The%20Berth%20Allocation%20with%20Tidal%20Windows%20Problem/P12-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 12. The Berth Allocation with Tidal Windows Problem
## Tier 1 — Mathematical Formulation

### Goal and scope
Assign each vessel to one berth and choose a service start time. A vessel may start only after it arrives, must have sufficient water depth, and must complete its service within a qualifying tidal window. Vessels assigned to the same berth must not overlap. The objective is to minimize total weighted time from arrival until service completion.

The notebook uses the supplied small example and a continuous assignment relaxation solved, when the instance is feasible, with SciPy's SLSQP constrained optimizer. The relaxation uses a penalty to encourage berth assignments near zero or one. This penalty is nonconvex, so SLSQP does not provide a global optimality guarantee. A successful solver status alone is not proof that the rounded schedule is feasible; feasibility is checked separately.

### Model assumptions
- The planning horizon is 24 hours, and all times are measured in hours from the beginning of that horizon.
- Tidal windows are known in advance. Within a listed window, the berth has its specified high-tide depth; outside the windows, it has its low-tide depth.
- A vessel's draft requirement must be met throughout service, and the complete service interval must lie within one listed tidal window.
- Vessel processing times and arrival times are deterministic. Berth lengths are recorded as input data but do not affect this example's constraints.
- Costs are linear in elapsed time from arrival until completion. The supplied demurrage values therefore act as weights, not as a separate charge that begins only after a grace period.

The model does not represent spatial berth positions, vessel-dependent handling rates, uncertain arrivals, changing tides within a window, or operational setup times. In particular, the requirement that the entire service fit within one tidal window is restrictive and is applied consistently in both the feasibility checks and the optimization constraints.

### Example data
The example contains three vessels, two berths, and two high-tide windows. Each window lasts three hours. Vessel A requires four hours of service and Vessel B requires six hours, so neither can complete service within either window. This makes the full instance infeasible under the stated assumptions. The notebook identifies this before attempting optimization rather than presenting an infeasible schedule as a solution.

### Solution sequence
1. Define vessel, berth, and tidal-window data.
2. Check whether each vessel has at least one individually feasible berth and start interval.
3. Define the weighted-completion-time objective, assignment constraints, tidal constraints, and berth non-overlap constraints.
4. If the feasibility screen passes, use SLSQP on the continuous assignment relaxation.
5. Interpret a solver result only after checking the extracted assignments and schedule against the original feasibility rules.

### Mathematical model

Let $V$ be the set of vessels, $B$ the set of berths, and $K$ the set of tidal windows. For vessel $v$, let $a_v$ be its arrival time, $p_v$ its processing time, $d_v$ its draft, and $w_v$ its cost weight. For berth $b$, let $D_{b,k}$ be its available depth during tidal window $k$. Window $k$ has start and end times $L_k$ and $U_k$.

The decision variable $x_{v,b}\in[0,1]$ represents the relaxed assignment of vessel $v$ to berth $b$; an integral assignment is either zero or one. The variable $s_v$ is the service start time. The objective is

$$\min_{x,s}\quad \sum_{v\in V} w_v\left(s_v+p_v-a_v\right) + 1000\sum_{v\in V}\sum_{b\in B}x_{v,b}(1-x_{v,b}).$$

The first term is weighted elapsed time from arrival to completion. The second term penalizes fractional berth assignments: it is zero at zero and one, and positive strictly between them. Because $x(1-x)$ is concave, this penalty makes the objective nonconvex; it encourages integral-looking assignments but does not make the relaxation a convex optimization problem.

Each vessel must be assigned exactly once:

$$\sum_{b\in B}x_{v,b}=1\quad(v\in V),\qquad 0\le x_{v,b}\le 1,\qquad a_v\le s_v\le 24.$$

A vessel can use berth $b$ in window $k$ only if the berth depth is sufficient and the complete service fits in that window. For an integral assignment, this means that some $k\in K$ satisfies

$$D_{b,k}\ge d_v,\qquad L_k\le s_v,\qquad s_v+p_v\le U_k.$$

For two vessels assigned to the same berth, their service intervals must be disjoint. For each pair $u<v$ and berth $b$, either $s_u+p_u\le s_v$ or $s_v+p_v\le s_u$. The code expresses this disjunction with a non-overlap constraint that is relaxed when the assignment variables are fractional.

In the implementation, `objective_function` computes the objective; `constraint_function` enforces the assignment equalities; and `inequality_constraint_function` represents the relaxed tidal and non-overlap conditions for SLSQP. `is_feasible_assignment` independently checks the original, non-relaxed tidal rule. The feasibility screen is necessary because a continuous relaxation must not be mistaken for a feasible integral schedule.

This formulation contains a disjunction for tidal windows and another for pairwise vessel order. SLSQP handles the resulting nonlinear constraints locally; it is not an exact mixed-integer method and does not guarantee a global optimum.

In [ ]:
# Import the numerical, plotting, and data tools used throughout the notebook.
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.optimize import minimize
from dataclasses import dataclass
from typing import List, Tuple
import pandas as pd

# Keep stochastic behavior reproducible if random operations are added to an exercise.
np.random.seed(42)

print("Libraries imported successfully.")

In [ ]:
@dataclass
class Vessel:
    """Store the operational requirements and cost weight of one vessel."""

    id: int
    draft: float
    processing_time: float
    arrival_time: float
    demurrage_cost: float


@dataclass
class Berth:
    """Store the dimensions and tide-dependent depths of one berth."""

    id: int
    length: float
    depth_low_tide: float
    depth_high_tide: float


@dataclass
class TidalWindow:
    """Store the time interval and depth factor for a tidal window."""

    start_time: float
    end_time: float
    depth_factor: float

In [ ]:
# Define the supplied three-vessel, two-berth example.
vessels = [
    Vessel(id=0, draft=12.0, processing_time=4.0, arrival_time=6.0, demurrage_cost=8000),
    Vessel(id=1, draft=14.0, processing_time=6.0, arrival_time=8.0, demurrage_cost=12000),
    Vessel(id=2, draft=11.0, processing_time=3.0, arrival_time=14.0, demurrage_cost=6000),
]

berths = [
    Berth(id=0, length=300.0, depth_low_tide=10.0, depth_high_tide=15.0),
    Berth(id=1, length=350.0, depth_low_tide=11.0, depth_high_tide=16.0),
]

# Each listed interval is modeled as a high-tide window.
tidal_windows = [
    TidalWindow(start_time=6.5, end_time=9.5, depth_factor=1.0),
    TidalWindow(start_time=19.0, end_time=22.0, depth_factor=1.0),
]

print(
    f"Problem setup: {len(vessels)} vessels, {len(berths)} berths, "
    f"{len(tidal_windows)} tidal windows."
)
print(
    "Vessels (id, draft in m, processing time in h, arrival time in h): "
    f"{[(v.id, v.draft, v.processing_time, v.arrival_time) for v in vessels]}"
)
print(
    "Berths (id, low-tide depth in m, high-tide depth in m): "
    f"{[(b.id, b.depth_low_tide, b.depth_high_tide) for b in berths]}"
)

In [ ]:
def get_tidal_depth(time: float, berth: Berth, tidal_windows: List[TidalWindow]) -> float:
    """Return the modeled berth depth at a time in the planning horizon."""
    for window in tidal_windows:
        if window.start_time <= time <= window.end_time:
            return berth.depth_high_tide

    return berth.depth_low_tide


def is_feasible_assignment(
    vessel: Vessel,
    berth: Berth,
    start_time: float,
    tidal_windows: List[TidalWindow],
    tolerance: float = 1e-8,
) -> bool:
    """Check arrival, draft, and full-service tidal-window feasibility."""
    if start_time < vessel.arrival_time - tolerance:
        return False

    end_time = start_time + vessel.processing_time
    for window in tidal_windows:
        if (
            berth.depth_high_tide + tolerance >= vessel.draft
            and window.start_time - tolerance <= start_time
            and end_time <= window.end_time + tolerance
        ):
            return True

    return False


def find_feasible_assignments(
    vessels: List[Vessel],
    berths: List[Berth],
    tidal_windows: List[TidalWindow],
) -> dict:
    """List feasible berth-window start intervals for each vessel."""
    feasible_intervals = {}
    for vessel in vessels:
        vessel_options = []
        for berth in berths:
            if berth.depth_high_tide < vessel.draft:
                continue
            for window in tidal_windows:
                earliest_start = max(vessel.arrival_time, window.start_time)
                latest_start = window.end_time - vessel.processing_time
                if earliest_start <= latest_start:
                    vessel_options.append(
                        {
                            "berth_id": berth.id,
                            "window_start": window.start_time,
                            "earliest_start": earliest_start,
                            "latest_start": latest_start,
                        }
                    )
        feasible_intervals[vessel.id] = vessel_options
    return feasible_intervals


feasible_intervals = find_feasible_assignments(vessels, berths, tidal_windows)
print("Individual vessel feasibility screen:")
for vessel in vessels:
    options = feasible_intervals[vessel.id]
    print(f"Vessel {vessel.id}: {len(options)} feasible berth-window start interval(s).")
    for option in options:
        print(
            f"  Berth {option['berth_id']}, window starting at "
            f"{option['window_start']:.1f} h: start in "
            f"[{option['earliest_start']:.1f}, {option['latest_start']:.1f}] h"
        )

### Feasibility screening
A feasible complete schedule must give every vessel at least one individually feasible berth-window interval. This is a necessary condition, though not sufficient: vessels may still conflict because they need the same berth at overlapping times.

The interval check is analytical rather than based on a time grid. For a vessel and a qualifying window, a valid start must be no earlier than both its arrival and the window start, and no later than the window end minus its processing time. If the resulting interval is empty, that vessel cannot use that berth-window combination.

In [ ]:
def objective_function(
    x: np.ndarray,
    vessels: List[Vessel],
    berths: List[Berth],
    assignment_penalty_weight: float = 1000.0,
) -> float:
    """Evaluate weighted completion cost plus the fractional-assignment penalty."""
    n_vessels = len(vessels)
    n_berths = len(berths)
    berth_assignments = x[: n_vessels * n_berths].reshape(n_vessels, n_berths)
    start_times = x[n_vessels * n_berths :]

    total_cost = 0.0
    for vessel_index, vessel in enumerate(vessels):
        completion_time = start_times[vessel_index] + vessel.processing_time
        total_cost += vessel.demurrage_cost * (completion_time - vessel.arrival_time)

        assignments = berth_assignments[vessel_index, :]
        total_cost += assignment_penalty_weight * np.sum(assignments * (1.0 - assignments))

    return float(total_cost)


def constraint_function(x: np.ndarray, vessels: List[Vessel], berths: List[Berth]) -> np.ndarray:
    """Return assignment equalities, which SLSQP requires to equal zero."""
    n_vessels = len(vessels)
    n_berths = len(berths)
    berth_assignments = x[: n_vessels * n_berths].reshape(n_vessels, n_berths)
    return np.sum(berth_assignments, axis=1) - 1.0


def inequality_constraint_function(
    x: np.ndarray,
    vessels: List[Vessel],
    berths: List[Berth],
    tidal_windows: List[TidalWindow],
    big_m: float = 48.0,
) -> np.ndarray:
    """Return tidal and berth non-overlap inequalities in SLSQP's g(x) >= 0 form."""
    n_vessels = len(vessels)
    n_berths = len(berths)
    berth_assignments = x[: n_vessels * n_berths].reshape(n_vessels, n_berths)
    start_times = x[n_vessels * n_berths :]
    inequalities = []

    # A selected berth-window combination must accommodate the full service interval.
    for vessel_index, vessel in enumerate(vessels):
        for berth_index, berth in enumerate(berths):
            qualifying_windows = [
                window
                for window in tidal_windows
                if berth.depth_high_tide >= vessel.draft
            ]
            margins = []
            for window in qualifying_windows:
                start = start_times[vessel_index]
                end = start + vessel.processing_time
                margins.append(
                    min(start - window.start_time, window.end_time - end)
                )

            # A finite negative margin keeps the constraint evaluable when no window qualifies.
            best_margin = max(margins) if margins else -big_m
            assignment = berth_assignments[vessel_index, berth_index]
            inequalities.append(best_margin + big_m * (1.0 - assignment))

    # If both relaxed assignments select the same berth, their intervals must not overlap.
    for berth_index in range(n_berths):
        for first in range(n_vessels):
            for second in range(first + 1, n_vessels):
                first_start = start_times[first]
                second_start = start_times[second]
                first_end = first_start + vessels[first].processing_time
                second_end = second_start + vessels[second].processing_time
                overlap_if_first_precedes = max(0.0, first_end - second_start)
                overlap_if_second_precedes = max(0.0, second_end - first_start)
                unavoidable_overlap = min(
                    overlap_if_first_precedes,
                    overlap_if_second_precedes,
                )
                first_assignment = berth_assignments[first, berth_index]
                second_assignment = berth_assignments[second, berth_index]
                inequalities.append(
                    big_m * (2.0 - first_assignment - second_assignment)
                    - unavoidable_overlap
                )

    return np.asarray(inequalities, dtype=float)


print("Objective and constraint functions are defined.")

### How the relaxation constraints work
SLSQP accepts equality constraints of the form $h(x)=0$ and inequality constraints of the form $g(x)\ge 0$. `constraint_function` returns one equality for each vessel, enforcing that its assignment values sum to one.

For an integral berth assignment, the tidal margin is nonnegative exactly when the service interval fits in at least one window with enough depth. The maximum over qualifying windows represents the choice of one feasible window. The `big_m` term relaxes this condition when the berth assignment is below one. Likewise, the non-overlap inequality permits either ordering of two services and is relaxed if the pair is not both assigned to that berth.

These are nonlinear, nonsmooth constraints, and the assignment variables are continuous during optimization. Consequently, the constraints describe a relaxation for fractional assignments; the final schedule still requires an independent feasibility check after extracting a berth for each vessel.

In [ ]:
n_vessels = len(vessels)
n_berths = len(berths)
n_variables = n_vessels * n_berths + n_vessels

# Use a round-robin berth pattern as the initial assignment point.
x0 = np.zeros(n_variables, dtype=float)
for vessel_index in range(n_vessels):
    berth_index = vessel_index % n_berths
    x0[vessel_index * n_berths + berth_index] = 1.0

# Arrival times provide bounded initial service-start values.
for vessel_index, vessel in enumerate(vessels):
    x0[n_vessels * n_berths + vessel_index] = vessel.arrival_time

bounds = [(0.0, 1.0)] * (n_vessels * n_berths)
bounds.extend((vessel.arrival_time, 24.0) for vessel in vessels)

constraints = [
    {
        "type": "eq",
        "fun": constraint_function,
        "args": (vessels, berths),
    },
    {
        "type": "ineq",
        "fun": inequality_constraint_function,
        "args": (vessels, berths, tidal_windows),
    },
]

print(f"Optimization setup: {n_variables} decision variables and {len(bounds)} bounds.")
print(
    "Initial objective value: "
    f"{objective_function(x0, vessels, berths):.2f}"
)

### Optimization and feasibility gate
The individual feasibility screen is a necessary condition for a full schedule. If any vessel has no feasible berth-window interval, optimizing the relaxed objective cannot produce a valid complete schedule. In that case the notebook reports infeasibility and does not present an optimizer output as a solution.

If every vessel passes the screen, the next cell runs the specified SLSQP local constrained optimization with the objective, bounds, and constraints defined above. Even then, solver success is only a numerical termination result; the extracted schedule must pass the original feasibility checks.

In [ ]:
instance_feasible = all(feasible_intervals[vessel.id] for vessel in vessels)
result = None

if instance_feasible:
    print("Running the SLSQP constrained optimization.")
    result = minimize(
        objective_function,
        x0,
        args=(vessels, berths),
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 1000, "ftol": 1e-6},
    )
    print(f"Optimization status: {result.message}")
    print(f"Success: {result.success}")
    print(f"Relaxed objective value: {result.fun:.2f}")
    print(f"Iterations: {result.nit}")
else:
    infeasible_vessels = [
        vessel.id for vessel in vessels if not feasible_intervals[vessel.id]
    ]
    print(
        "No complete feasible schedule exists under the stated tidal-window "
        "and full-service assumptions."
    )
    print(f"Vessels with no individually feasible berth-window interval: {infeasible_vessels}")
    print("The optimizer is not run because the feasibility screen fails.")

In [ ]:
def extract_solution(
    x: np.ndarray,
    vessels: List[Vessel],
    berths: List[Berth],
) -> List[dict]:
    """Convert optimization variables into one reported berth per vessel."""
    n_vessels = len(vessels)
    n_berths = len(berths)
    berth_assignments = x[: n_vessels * n_berths].reshape(n_vessels, n_berths)
    start_times = x[n_vessels * n_berths :]

    solution = []
    for vessel_index, vessel in enumerate(vessels):
        assigned_berth = int(np.argmax(berth_assignments[vessel_index, :]))
        confidence = float(berth_assignments[vessel_index, assigned_berth])
        start_time = float(start_times[vessel_index])
        solution.append(
            {
                "vessel_id": vessel.id,
                "assigned_berth": assigned_berth,
                "assignment_confidence": confidence,
                "start_time": start_time,
                "end_time": start_time + vessel.processing_time,
                "processing_time": vessel.processing_time,
                "draft": vessel.draft,
                "demurrage_cost": vessel.demurrage_cost,
            }
        )

    return solution


def validate_solution(
    solution: List[dict],
    vessels: List[Vessel],
    berths: List[Berth],
    tidal_windows: List[TidalWindow],
    tolerance: float = 1e-6,
) -> Tuple[bool, List[str]]:
    """Check extracted assignments, tidal feasibility, and berth non-overlap."""
    violations = []
    solution_by_id = {item["vessel_id"]: item for item in solution}

    for vessel in vessels:
        item = solution_by_id[vessel.id]
        berth = berths[item["assigned_berth"]]
        if not np.isclose(item["assignment_confidence"], 1.0, atol=tolerance):
            violations.append(f"Vessel {vessel.id} has a fractional berth assignment.")
        if not is_feasible_assignment(
            vessel, berth, item["start_time"], tidal_windows, tolerance
        ):
            violations.append(f"Vessel {vessel.id} violates an arrival, draft, or tidal rule.")

    for berth in berths:
        berth_items = [
            item for item in solution if item["assigned_berth"] == berth.id
        ]
        berth_items.sort(key=lambda item: item["start_time"])
        for first, second in zip(berth_items, berth_items[1:]):
            if first["end_time"] > second["start_time"] + tolerance:
                violations.append(
                    f"Vessels {first['vessel_id']} and {second['vessel_id']} overlap "
                    f"at Berth {berth.id}."
                )

    return not violations, violations


solution = []
if result is not None and result.success:
    solution = extract_solution(result.x, vessels, berths)
    schedule_is_feasible, validation_violations = validate_solution(
        solution, vessels, berths, tidal_windows
    )

    if schedule_is_feasible:
        print("Feasible extracted schedule:")
        total_cost = 0.0
        for item in solution:
            vessel = vessels[item["vessel_id"]]
            cost = vessel.demurrage_cost * (item["end_time"] - vessel.arrival_time)
            total_cost += cost
            print(
                f"Vessel {vessel.id}: Berth {item['assigned_berth']}, "
                f"start {item['start_time']:.2f} h, "
                f"end {item['end_time']:.2f} h, weighted cost {cost:,.0f}"
            )
        print(f"Total weighted completion cost: {total_cost:,.0f}")
    else:
        print("The numerical result does not yield a feasible extracted schedule:")
        for violation in validation_violations:
            print(f"- {violation}")
elif result is not None:
    print("SLSQP did not report successful termination; no schedule is reported.")
else:
    print("No schedule is extracted because the instance fails the feasibility screen.")

### Reading the solution report
The optimizer's continuous assignment values are converted to a reported berth by selecting the largest assignment value. The corresponding confidence value indicates how close that assignment is to one; it is not a probability. A fractional assignment is not a valid physical berth choice.

The validation step checks the selected berth, arrival time, draft depth, full-service tidal-window condition, and pairwise non-overlap. Only a schedule that passes every check is reported as feasible. The objective value from the relaxed optimization and the weighted cost recomputed from a validated schedule are distinct quantities when assignments remain fractional.

In [ ]:
def visualize_solution(
    solution: List[dict],
    vessels: List[Vessel],
    berths: List[Berth],
    tidal_windows: List[TidalWindow],
) -> None:
    """Plot the berth schedule and modeled depth over the 24-hour horizon."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4", "#FFEAA7"]

    ax1.set_title("Berth Allocation Schedule and Tidal Windows")
    ax1.set_xlabel("Time (hours from horizon start)")
    ax1.set_ylabel("Berth")
    ax1.set_xlim(0, 24)
    ax1.set_ylim(-0.5, len(berths) - 0.5)
    ax1.set_yticks(range(len(berths)))
    ax1.set_yticklabels([f"Berth {berth.id}" for berth in berths])
    ax1.grid(True, alpha=0.3)

    for index, window in enumerate(tidal_windows):
        ax1.axvspan(
            window.start_time,
            window.end_time,
            alpha=0.2,
            color="lightblue",
            label="High-tide window" if index == 0 else None,
        )

    for item in solution:
        vessel_id = item["vessel_id"]
        berth_id = item["assigned_berth"]
        color = colors[vessel_id % len(colors)]
        rectangle = patches.Rectangle(
            (item["start_time"], berth_id - 0.3),
            item["end_time"] - item["start_time"],
            0.6,
            linewidth=1.5,
            edgecolor="black",
            facecolor=color,
            alpha=0.75,
            label=f"Vessel {vessel_id}",
        )
        ax1.add_patch(rectangle)
        ax1.text(
            (item["start_time"] + item["end_time"]) / 2,
            berth_id,
            f"V{vessel_id}",
            ha="center",
            va="center",
            fontweight="bold",
        )

    for index, vessel in enumerate(vessels):
        ax1.axvline(
            x=vessel.arrival_time,
            color="red",
            linestyle="--",
            alpha=0.5,
            label="Arrival time" if index == 0 else None,
        )
    ax1.legend(loc="upper right")

    ax2.set_title("Vessel Draft Requirements and Berth Depth")
    ax2.set_xlabel("Time (hours from horizon start)")
    ax2.set_ylabel("Depth (meters)")
    ax2.set_xlim(0, 24)
    ax2.grid(True, alpha=0.3)

    time_points = np.linspace(0, 24, 481)
    for berth in berths:
        depths = [
            get_tidal_depth(time, berth, tidal_windows) for time in time_points
        ]
        ax2.step(
            time_points,
            depths,
            where="post",
            linewidth=2,
            label=f"Berth {berth.id} available depth",
        )

    for item in solution:
        vessel = vessels[item["vessel_id"]]
        color = colors[vessel.id % len(colors)]
        ax2.hlines(
            vessel.draft,
            item["start_time"],
            item["end_time"],
            colors=color,
            linewidth=3,
            label=f"Vessel {vessel.id} draft requirement",
        )

    ax2.legend(loc="best")
    plt.tight_layout()
    plt.show()


if solution:
    visualize_solution(solution, vessels, berths, tidal_windows)
else:
    print("No feasible schedule is available to visualize.")

In [ ]:
def analyze_solution_quality(
    solution: List[dict],
    vessels: List[Vessel],
    berths: List[Berth],
    tidal_windows: List[TidalWindow],
) -> None:
    """Report feasibility, berth utilization, weighted cost, and window use."""
    if not solution:
        print("Solution quality cannot be analyzed because no schedule is available.")
        return

    is_feasible, violations = validate_solution(
        solution, vessels, berths, tidal_windows
    )
    feasible_count = sum(
        is_feasible_assignment(
            vessel,
            berths[solution_item["assigned_berth"]],
            solution_item["start_time"],
            tidal_windows,
        )
        for vessel in vessels
        for solution_item in solution
        if solution_item["vessel_id"] == vessel.id
    )

    print("Solution Quality Analysis")
    print("=" * 40)
    print(f"Individually feasible vessel assignments: {feasible_count}/{len(vessels)}")
    print(f"Complete schedule feasible: {is_feasible}")
    if violations:
        for violation in violations:
            print(f"- {violation}")

    print("\nBerth utilization over the 24-hour horizon:")
    for berth in berths:
        occupied_time = sum(
            item["processing_time"]
            for item in solution
            if item["assigned_berth"] == berth.id
        )
        utilization = occupied_time / 24.0 * 100.0
        print(
            f"Berth {berth.id}: {utilization:.1f}% "
            f"({occupied_time:.1f} service hours)"
        )

    total_cost = sum(
        vessel.demurrage_cost
        * (solution_item["end_time"] - vessel.arrival_time)
        for vessel in vessels
        for solution_item in solution
        if solution_item["vessel_id"] == vessel.id
    )
    print("\nWeighted completion cost:")
    print(f"Total: {total_cost:,.0f}")
    print(f"Average per vessel: {total_cost / len(vessels):,.0f}")

    print("\nTidal-window service utilization:")
    for window_index, window in enumerate(tidal_windows, start=1):
        duration = window.end_time - window.start_time
        utilized_time = sum(
            item["processing_time"]
            for item in solution
            if window.start_time <= item["start_time"]
            and item["end_time"] <= window.end_time
        )
        efficiency = utilized_time / duration * 100.0 if duration > 0 else 0.0
        print(
            f"Window {window_index}: {efficiency:.1f}% "
            f"({utilized_time:.1f}/{duration:.1f} service hours)"
        )


analyze_solution_quality(solution, vessels, berths, tidal_windows)

## Interpreting the method and its limits

### What this formulation provides
- A transparent objective that weights each vessel's elapsed time from arrival to completion.
- Explicit assignment, arrival-time, tidal-depth, full-window, and berth non-overlap requirements.
- A numerical SLSQP implementation of a continuous assignment relaxation, with independent checks on any extracted schedule.
- A feasibility screen that explains why an instance cannot have a complete schedule under the stated assumptions.

### What it does not establish
- The relaxation is nonconvex because of the fractional-assignment penalty and the disjunctive constraints.
- SLSQP is a local nonlinear optimizer. Its successful termination does not establish global optimality, and the relaxed assignment values may not be integral.
- The feasibility screen detects an individually impossible vessel, but passing it does not prove that vessels can be scheduled together without berth conflicts.
- The supplied instance is infeasible under the rule that every service must fit entirely inside one listed three-hour tidal window: the first two vessels require longer services than that window permits. Thus there is no valid complete schedule for these data under this model.

For this instance, the appropriate result is the infeasibility diagnosis, not a numerical schedule or an optimality claim. The model can be studied on other data only when the stated assumptions and all feasibility rules remain explicit.